# Test Model
This will be a simple CNN that does not use any pretrained models

In [7]:
# Package loading
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd

## Image loading

In [2]:

all_images = []
img_directory = 'molecule_images/'

# Runs from 1 to 1547(the greatest numbered img) and will throw errors but continue on missing file names.
for i in range(1, 1548):
    try:
        img = tf.io.read_file('molecule_images/BACE_' + str(i) + '.png')
        all_images.append(tf.io.decode_png(img))
    except:
        pass

## Read in data to extract target features and create train, validation, test split

In [20]:
# Extract target variable
bace_data = pd.read_csv('bace.csv')
pIC = bace_data['pIC50']

# Split into training and test/validation sets
X_train, X_tval, y_train, y_tval = train_test_split(all_images, pIC, test_size = 0.2, random_state = 42)

# Split test and validation sets
X_val, X_test, y_val, y_test = train_test_split(X_tval, y_tval, test_size = 0.5, random_state = 42)

# Convert X's to tensors to avoid later errors
X_train = tf.convert_to_tensor(X_train)
X_test = tf.convert_to_tensor(X_test)
X_val = tf.convert_to_tensor(X_val)

## Initial Model Construction

In [33]:
inputs = keras.Input(shape = (400, 400, 3))

x = layers.Resizing(180, 180)(inputs)
x = layers.Rescaling(1./255)(x)
x = layers.Conv2D(filters = 32, kernel_size = 3, activation = 'relu')(x)
x = layers.MaxPool2D(pool_size = 2)(x)
x = layers.Conv2D(filters = 64, kernel_size = 3, activation = 'relu')(x)
x = layers.MaxPool2D(pool_size = 2)(x)
x = layers.Conv2D(filters = 128, kernel_size = 3, activation = 'relu')(x)
x = layers.MaxPool2D(pool_size = 2)(x)
x = layers.Conv2D(filters = 256, kernel_size = 3, activation = 'relu')(x)
x = layers.Conv2D(filters = 256, kernel_size = 3, activation = 'relu')(x)
x = layers.Flatten()(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(1, activation = "linear")(x)
model = keras.Model(inputs = inputs, outputs = outputs)

model.compile(loss = "mse",
              optimizer = "adam",
              metrics = ["mae"])



In [34]:
model.fit(X_train, y_train, epochs = 50, validation_data = (X_val, y_val), batch_size = 32)

Epoch 1/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 10s 150ms/step - loss: 21.5169 - mae: 3.6377 - val_loss: 2.4461 - val_mae: 1.3560
Epoch 2/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step - loss: 1.9637 - mae: 1.1405 - val_loss: 1.6941 - val_mae: 1.0784
Epoch 3/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - loss: 1.8353 - mae: 1.1257 - val_loss: 1.7991 - val_mae: 1.0577
Epoch 4/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - loss: 1.8229 - mae: 1.1105 - val_loss: 1.6377 - val_mae: 1.0520
Epoch 5/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step - loss: 1.7511 - mae: 1.0954 - val_loss: 1.6874 - val_mae: 1.0249
Epoch 6/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step - loss: 1.8036 - mae: 1.1025 - val_loss: 1.4965 - val_mae: 1.0085
Epoch 7/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - loss: 1.7331 - mae: 1.0710 - val_loss: 1.7833 - val_mae: 1.0160
Epoch 8/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 48ms/step - loss: 1.7414 - mae: 1.0766 - val_loss: 1.3887 - val_mae: 0.9805
Epoch 9/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - loss: 1.

In [35]:
model.evaluate(X_test, y_test)

5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 436ms/step - loss: 0.8309 - mae: 0.7208


[0.8309236764907837, 0.7208148837089539]